In [1]:
import pandas as pd

#Load the dataset into a pandas DataFrame using pd.read_csv()
df=pd.read_csv("/content/train_bigmart.csv")
print("\nDataset has been loaded successfully into Pandas DataFrame")

#Print the first five rows
print("\nFirst Five rows:")
print(df.head())

#Print the column Datatypes
print("\nColumn Datatypes:")
print(df.dtypes)

#Print the shape of the dataframe
print("\nDataframe Shape:",df.shape)


Dataset has been loaded successfully into Pandas DataFrame

First Five rows:
  Item_Identifier  Item_Weight Item_Fat_Content  Item_Visibility  \
0           FDA15         9.30          Low Fat         0.016047   
1           DRC01         5.92          Regular         0.019278   
2           FDN15        17.50          Low Fat         0.016760   
3           FDX07        19.20          Regular         0.000000   
4           NCD19         8.93          Low Fat         0.000000   

               Item_Type  Item_MRP Outlet_Identifier  \
0                  Dairy  249.8092            OUT049   
1            Soft Drinks   48.2692            OUT018   
2                   Meat  141.6180            OUT049   
3  Fruits and Vegetables  182.0950            OUT010   
4              Household   53.8614            OUT013   

   Outlet_Establishment_Year Outlet_Size Outlet_Location_Type  \
0                       1999      Medium               Tier 1   
1                       2009      Medium      

In [4]:
import numpy as np
#Null value analysis
null_counts= df.isnull().sum()

null_percentage=(df.isnull().sum() / df.shape[0])*100

null_report= pd.DataFrame({'Missing_count':null_counts,
                           'Missing_Percentage':null_percentage})

print(null_report)

#Reportcolumns exceed a 20% null rate
high_nulls = null_report[null_report['Missing_Percentage'] > 20]
print("\nColumns exceeding 20% null rate:\n", high_nulls)

numeric_cols_stage1 = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumeric columns at this stage:", numeric_cols_stage1)
print("Skewness of these columns is computed in Task 5; the two highest-skew columns")
print("will have their imputation deferred to Task 7a for the before/after comparison.)")



                           Missing_count  Missing_Percentage
Item_Identifier                        0            0.000000
Item_Weight                         1463           17.165317
Item_Fat_Content                       0            0.000000
Item_Visibility                        0            0.000000
Item_Type                              0            0.000000
Item_MRP                               0            0.000000
Outlet_Identifier                      0            0.000000
Outlet_Establishment_Year              0            0.000000
Outlet_Size                         2410           28.276428
Outlet_Location_Type                   0            0.000000
Outlet_Type                            0            0.000000
Item_Outlet_Sales                      0            0.000000

Columns exceeding 20% null rate:
              Missing_count  Missing_Percentage
Outlet_Size           2410           28.276428

Numeric columns at this stage: ['Item_Weight', 'Item_Visibility', 'Item_MRP',

In [3]:
#Duplicate detection and removal

dup_count=df.duplicated().sum()
print("\nDuplicate count:",dup_count)

#Remove the duplicates
df_clean=df.drop_duplicates()
print("\nShape after removing exact duplicates:", df.shape)

#Report null percentage changes

null_counts_before= df.isnull().sum()

null_percentage_before=(df.isnull().sum() / df.shape[0])*100

null_counts_after= df_clean.isnull().sum()

null_percentage_after=(df_clean.isnull().sum() / df_clean.shape[0])*100

null_comparison=pd.DataFrame({'Missing_Before':null_percentage_before,
                             'Missing_after':null_percentage_after})
print(null_comparison)




Duplicate count: 0

Shape after removing exact duplicates: (8523, 12)
                           Missing_Before  Missing_after
Item_Identifier                  0.000000       0.000000
Item_Weight                      0.000000       0.000000
Item_Fat_Content                 0.000000       0.000000
Item_Visibility                  0.000000       0.000000
Item_Type                        0.000000       0.000000
Item_MRP                         0.000000       0.000000
Outlet_Identifier                0.000000       0.000000
Outlet_Establishment_Year        0.000000       0.000000
Outlet_Size                     28.276428      28.276428
Outlet_Location_Type             0.000000       0.000000
Outlet_Type                      0.000000       0.000000
Item_Outlet_Sales                0.000000       0.000000


In [7]:
# Memory usage before conversion
mem_before = df.memory_usage(deep=True).sum()

# # (a) Item_Fat_Content is a numeric-in-spirit categorical field stored inconsistently
# as free-text object with typo/abbreviation variants ('low fat','LF','reg') for what
# should be exactly 2 categories. This is the "incorrect inferred representation" fix:
# standardize the labels, THEN cast to category dtype.

df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
})
print("Item_Fat_Content cleaned unique values:", sorted(df["Item_Fat_Content"].unique()))

# (b) Outlet_Establishment_Year is stored as int64 but is really a calendar year used
# as an identifier/date, not a quantity to sum or average meaningfully on its own.
# Convert it into an interpretable numeric feature: Outlet_Age (dataset compiled 2013).

df["Outlet_Age"] = 2013 - df["Outlet_Establishment_Year"]
print("\nEngineered numeric feature 'Outlet_Age' = 2013 - Outlet_Establishment_Year.")

# (c) Repetitive string columns -> category dtype
categorical_cols = ['Item_Fat_Content', 'Item_Type','Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# After conversion
mem_after = df.memory_usage(deep=True).sum()

print("\nMemory usage before:", mem_before, "bytes")
print("\nMemory usage after:", mem_after, "bytes")


Item_Fat_Content cleaned unique values: ['Low Fat', 'Regular']

Engineered numeric feature 'Outlet_Age' = 2013 - Outlet_Establishment_Year.

Memory usage before: 924162 bytes

Memory usage after: 924162 bytes


In [13]:
#Descriptive statistics and skewness

numeric_cols = ['Item_Weight','Item_Visibility', 'Item_MRP','Item_Outlet_Sales']  # Outlet_Establishment_Year excluded (identifier, not a measurement)
print(df[numeric_cols].describe())

skew_vals = df[numeric_cols].apply(lambda s: s.skew())
print("\nSkewness per numeric column:")
print(skew_vals)

most_skewed_col = skew_vals.abs().idxmax()
print(f"\nColumn with highest absolute skewness: '{most_skewed_col}' (skew = {skew_vals[most_skewed_col]:.3f})")

# Now fill Item_Weight's nulls -- it is NOT one of the two highest-skew columns
# so it's safe to median-fill now per Task 2.
df["Item_Weight"] = df["Item_Weight"].fillna(df["Item_Weight"].median())

# Outlet_Size (categorical, 28.3% null - exceeds 20% threshold) is filled with the
# per Outlet_Type mode rather than dropped, since the missingness is structural
# (specific outlets never reported size) not random, and dropping it would lose 28%
# of rows. A 'Missing' category is also viable; here we use the outlet-type mode.

outlet_size_mode_by_type = df.groupby("Outlet_Type", observed=True)["Outlet_Size"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else "Small"
)
print("\nOutlet_Size mode by Outlet_Type (used for imputation):")
print(outlet_size_mode_by_type)

df["Outlet_Size"] = df.apply(
    lambda r: outlet_size_mode_by_type[r["Outlet_Type"]] if pd.isnull(r["Outlet_Size"]) else r["Outlet_Size"],
    axis=1
).astype("category")
print("Outlet_Size nulls remaining:", df["Outlet_Size"].isnull().sum())

       Item_Weight  Item_Visibility     Item_MRP  Item_Outlet_Sales
count   8523.00000      8523.000000  8523.000000        8523.000000
mean      12.81342         0.066132   140.992782        2181.288914
std        4.22724         0.051598    62.275067        1706.499616
min        4.55500         0.000000    31.290000          33.290000
25%        9.31000         0.026989    93.826500         834.247400
50%       12.60000         0.053931   143.012800        1794.331000
75%       16.00000         0.094585   185.643700        3101.296400
max       21.35000         0.328391   266.888400       13086.964800

Skewness per numeric column:
Item_Weight          0.121845
Item_Visibility      1.167091
Item_MRP             0.127202
Item_Outlet_Sales    1.177531
dtype: float64

Column with highest absolute skewness: 'Item_Outlet_Sales' (skew = 1.178)

Outlet_Size mode by Outlet_Type (used for imputation):
Outlet_Type
Grocery Store         Small
Supermarket Type1     Small
Supermarket Type2    Med

In [15]:
#Outlier detection with IQR:
numeric_cols = [ 'Item_Weight','Item_Visibility','Item_MRP','Item_Outlet_Sales']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}:")
    print(f"  Q1 = {Q1:.3f}, Q3 = {Q3:.3f}, IQR = {IQR:.3f} Lower Bound = {lower_bound:.3f}, Upper Bound = {upper_bound:.3f}")
    print(f"  Outlier count = {outliers.shape[0]}")


Item_Weight:
  Q1 = 9.310, Q3 = 16.000, IQR = 6.690 Lower Bound = -0.725, Upper Bound = 26.035
  Outlier count = 0
Item_Visibility:
  Q1 = 0.027, Q3 = 0.095, IQR = 0.068 Lower Bound = -0.074, Upper Bound = 0.196
  Outlier count = 144
Item_MRP:
  Q1 = 93.826, Q3 = 185.644, IQR = 91.817 Lower Bound = -43.899, Upper Bound = 323.370
  Outlier count = 0
Item_Outlet_Sales:
  Q1 = 834.247, Q3 = 3101.296, IQR = 2267.049 Lower Bound = -2566.326, Upper Bound = 6501.870
  Outlier count = 186


In [25]:
import matplotlib.pyplot as plt
# 1. Line plot: Item_Outlet_Sales sorted by row index
plt.figure(figsize=(11, 5))
sales_sorted = df["Item_Outlet_Sales"].reset_index(drop=True)
plt.plot(sales_sorted.index, sales_sorted.values, linewidth=0.6, color="#2c6e91")
plt.title("Item_Outlet_Sales by Row Index")
plt.xlabel("Row Index")
plt.ylabel("Item Outlet Sales ($)")
plt.tight_layout()
PLOTS = "/content/sample_data/PLOTS"
plt.savefig(f"{PLOTS}/1_line_sales_by_index.png", dpi=110)
plt.close()

In [26]:
# 2. Bar chart: mean Item_Outlet_Sales by Outlet_Type
plt.figure(figsize=(8, 5))
mean_sales_type = df.groupby("Outlet_Type", observed=True)["Item_Outlet_Sales"].mean().sort_values(ascending=False)
plt.bar(mean_sales_type.index, mean_sales_type.values, color=["#2c6e91", "#e07a3e", "#4c9a5b", "#c94c4c"])
plt.title("Mean Item_Outlet_Sales by Outlet_Type")
plt.xlabel("Outlet Type")
plt.ylabel("Mean Sales ($)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{PLOTS}/2_bar_mean_sales_by_outlettype.png", dpi=110)
plt.close()


In [28]:
# 3. Histogram of most skewed column
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.histplot(df[most_skewed_col].dropna(), bins=20, color="#8a4fa3")
plt.title(f"Distribution of {most_skewed_col} (most skewed numeric column)")
plt.xlabel(most_skewed_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"{PLOTS}/3_histogram_most_skewed.png", dpi=110)
plt.close()



In [29]:
# 4. Scatter plot: Item_MRP vs Item_Outlet_Sales (expected positive relationship - higher priced items sell for more)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="Item_MRP", y="Item_Outlet_Sales", alpha=0.4, s=18, color="#2c6e91")
plt.title("Item_Outlet_Sales vs. Item_MRP")
plt.xlabel("Item MRP ($)")
plt.ylabel("Item Outlet Sales ($)")
plt.tight_layout()
plt.savefig(f"{PLOTS}/4_scatter_sales_vs_mrp.png", dpi=110)
plt.close()

In [30]:
# 5. Box plot: Item_Outlet_Sales by Outlet_Size
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Outlet_Size", y="Item_Outlet_Sales",
            order=["Small", "Medium", "High"])
plt.title("Item_Outlet_Sales Distribution by Outlet_Size")
plt.xlabel("Outlet Size")
plt.ylabel("Item Outlet Sales ($)")
plt.tight_layout()
plt.savefig(f"{PLOTS}/5_boxplot_sales_by_outletsize.png", dpi=110)
plt.close()

In [31]:
#Correlation heat map:
pearson_corr = df[numeric_cols].corr(method="pearson")
print(pearson_corr)

plt.figure(figsize=(7, 6))
sns.heatmap(pearson_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Pearson Correlation Heat Map (Numeric Columns)")
plt.tight_layout()
plt.savefig(f"{PLOTS}/6_heatmap_pearson.png", dpi=110)
plt.close()

corr_abs = pearson_corr.abs().where(~np.eye(len(numeric_cols), dtype=bool))
max_pair = corr_abs.stack().idxmax()
print(f"\nHighest absolute correlation pair: {max_pair} = {pearson_corr.loc[max_pair]:.3f}")


                   Item_Weight  Item_Visibility  Item_MRP  Item_Outlet_Sales
Item_Weight           1.000000        -0.014168  0.024951           0.009693
Item_Visibility      -0.014168         1.000000 -0.001315          -0.128625
Item_MRP              0.024951        -0.001315  1.000000           0.567574
Item_Outlet_Sales     0.009693        -0.128625  0.567574           1.000000

Highest absolute correlation pair: ('Item_MRP', 'Item_Outlet_Sales') = 0.568


In [32]:
#Imputation strategy comparison:
top2_skew_cols = skew_vals.abs().sort_values(ascending=False).index[:2].tolist()
print("Two highest-skewness numeric columns:", top2_skew_cols)

mean_median_table = pd.DataFrame({
    "mean_before_imputation": [df[c].mean() for c in top2_skew_cols],
    "median_before_imputation": [df[c].median() for c in top2_skew_cols],
    "skew": [skew_vals[c] for c in top2_skew_cols],
    "nulls_before": [df[c].isnull().sum() for c in top2_skew_cols],
}, index=top2_skew_cols)
print(mean_median_table)

for c in top2_skew_cols:
    df[c] = df[c].fillna(df[c].median())

print("\nNulls remaining after imputation:")
print(df[top2_skew_cols].isnull().sum())


Two highest-skewness numeric columns: ['Item_Outlet_Sales', 'Item_Visibility']
                   mean_before_imputation  median_before_imputation      skew  \
Item_Outlet_Sales             2181.288914               1794.331000  1.177531   
Item_Visibility                  0.066132                  0.053931  1.167091   

                   nulls_before  
Item_Outlet_Sales             0  
Item_Visibility               0  

Nulls remaining after imputation:
Item_Outlet_Sales    0
Item_Visibility      0
dtype: int64


In [33]:
#Spearman rank correlation
spearman_corr = df[numeric_cols].corr(method="spearman")
print("Spearman correlation matrix:")
print(spearman_corr)
print("\nPearson correlation matrix (from Task 7):")
print(pearson_corr)

diff = (spearman_corr - pearson_corr).abs()
diff_pairs = diff.where(~np.eye(len(numeric_cols), dtype=bool)).stack().sort_values(ascending=False)
seen = set()
top_diffs = []
for (a, b), v in diff_pairs.items():
    key = frozenset((a, b))
    if key in seen:
        continue
    seen.add(key)
    top_diffs.append((a, b, v, spearman_corr.loc[a, b], pearson_corr.loc[a, b]))
    if len(top_diffs) == 3:
        break

diff_table = pd.DataFrame(top_diffs, columns=["col_a", "col_b", "abs_diff", "spearman", "pearson"])
print("\nTop 3 pairs by |Spearman - Pearson|:")
print(diff_table)


Spearman correlation matrix:
                   Item_Weight  Item_Visibility  Item_MRP  Item_Outlet_Sales
Item_Weight           1.000000        -0.008316  0.028555           0.014486
Item_Visibility      -0.008316         1.000000  0.005688          -0.115076
Item_MRP              0.028555         0.005688  1.000000           0.562986
Item_Outlet_Sales     0.014486        -0.115076  0.562986           1.000000

Pearson correlation matrix (from Task 7):
                   Item_Weight  Item_Visibility  Item_MRP  Item_Outlet_Sales
Item_Weight           1.000000        -0.014168  0.024951           0.009693
Item_Visibility      -0.014168         1.000000 -0.001315          -0.128625
Item_MRP              0.024951        -0.001315  1.000000           0.567574
Item_Outlet_Sales     0.009693        -0.128625  0.567574           1.000000

Top 3 pairs by |Spearman - Pearson|:
               col_a            col_b  abs_diff  spearman   pearson
0  Item_Outlet_Sales  Item_Visibility  0.013549 -0.1

In [34]:
#Grouped aggregation
group_col, agg_col = "Outlet_Type", "Item_Outlet_Sales"
grouped = df.groupby(group_col, observed=True)[agg_col].agg(["mean", "std", "count"])
print(grouped)

highest_mean_group = grouped["mean"].idxmax()
highest_std_group = grouped["std"].idxmax()
mean_ratio = grouped["mean"].max() / grouped["mean"].min()
print(f"\nHighest mean group: {highest_mean_group} (mean={grouped['mean'].max():.2f})")
print(f"Highest std group: {highest_std_group} (std={grouped['std'].max():.2f})")
print(f"Ratio of highest to lowest group mean: {mean_ratio:.3f}")

                          mean          std  count
Outlet_Type                                       
Grocery Store       339.828500   260.851582   1083
Supermarket Type1  2316.181148  1515.965558   5577
Supermarket Type2  1995.498739  1375.932889    928
Supermarket Type3  3694.038558  2127.760054    935

Highest mean group: Supermarket Type3 (mean=3694.04)
Highest std group: Supermarket Type3 (std=2127.76)
Ratio of highest to lowest group mean: 10.870


In [35]:
#Save the clean dataset
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv with shape:", df.shape)

Saved cleaned_data.csv with shape: (8523, 13)
